In [2]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

In [7]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

url = "https://www.ambitionbox.com/list-of-companies?campaign=desktop_nav"

response = requests.get(url, headers=headers)
webpage = response.text

In [8]:
soup = BeautifulSoup(webpage,'lxml')

In [ ]:
print(soup.prettify())

In [10]:
company = soup.find_all('div',class_='companyCardWrapper')
com_names = pd.DataFrame()

In [11]:
name = []
rating = []
types = []
reviews = []
salary = []
job = []
interview = []

for i in company:
    
    name.append(i.find('h2').get_text(strip=True))
    
    rating.append(i.find('div', class_='rating_text').get_text(strip=True))
    
    types.append(i.find('span', class_='companyCardWrapper__interLinking').get_text(strip=True))
    
    titles = i.find_all('span', class_='companyCardWrapper__ActionTitle')
    counts = i.find_all('span', class_='companyCardWrapper__ActionCount')
    
    for t, c in zip(titles, counts):
        key = t.get_text(strip=True)
        value = c.get_text(strip=True)
        
        if 'Reviews' in key:
            reviews.append(value)
        elif 'Salaries' in key:
            salary.append(value)
        elif 'Jobs' in key:
            job.append(value)
        elif 'Interviews' in key:
            interview.append(value)
    

In [12]:
types

['IT Services & Consulting | Bengaluru +441 other locations',
 'IT Services & Consulting | Bengaluru +259 other locations',
 'IT Services & Consulting | Hyderabad +373 other locations',
 'IT Services & Consulting | Hyderabad +231 other locations',
 'IT Services & Consulting | Bengaluru +180 other locations',
 'Banking | Mumbai +1844 other locations',
 'IT Services & Consulting | Bengaluru +243 other locations',
 'IT Services & Consulting | Chennai +231 other locations',
 'Banking | Mumbai +1442 other locations',
 'IT Services & Consulting | Hyderabad +322 other locations',
 'Analytics & KPO | Hyderabad +181 other locations',
 'BPO | Mumbai +254 other locations',
 'Telecom | Mumbai +1956 other locations',
 'Banking | Mumbai +1516 other locations',
 'BPO | Bengaluru +181 other locations',
 'Internet | Bengaluru +515 other locations',
 'Retail | Mumbai +1157 other locations',
 'BPO | Noida +51 other locations',
 'IT Services & Consulting | Bengaluru +144 other locations',
 'IT Services & 

In [13]:
d = {'name':name, 'rating':rating, 'types':types, 'reviews':reviews, 'salary':salary, 'jobs':job, 'interviews':interview}

In [14]:
df = pd.DataFrame(d)

In [15]:
df.head(2)

,name,rating,types,reviews,salary,jobs,interviews
0,TCS,3.3,IT Services & Consulting | Bengaluru +441 othe...,1.1L,10.1L,4.4k,11.8k
1,Accenture,3.7,IT Services & Consulting | Bengaluru +259 othe...,73.1k,6.9L,36.8k,9.3k


In [16]:
def convert(t):
    if '|' in t:
        return t.split('|')[1].strip()
    return None

In [17]:
def convert(a):
    if '|' in a:
        return a.split('|')[0].strip()
    return None

In [18]:
df['Locations'] = df['types'].apply(convert)

In [19]:
df['types'] = df['types'].apply(convert)

In [20]:
df.head(10)

,name,rating,types,reviews,salary,jobs,interviews,Locations
0,TCS,3.3,IT Services & Consulting,1.1L,10.1L,4.4k,11.8k,IT Services & Consulting
1,Accenture,3.7,IT Services & Consulting,73.1k,6.9L,36.8k,9.3k,IT Services & Consulting
2,Wipro,3.6,IT Services & Consulting,64.8k,4.9L,367,6.8k,IT Services & Consulting
3,Cognizant,3.7,IT Services & Consulting,61.2k,6.1L,466,6.4k,IT Services & Consulting
4,Capgemini,3.6,IT Services & Consulting,53k,4.9L,2.1k,5.5k,IT Services & Consulting
5,HDFC Bank,3.8,Banking,52k,1.5L,295,3k,Banking
6,Infosys,3.5,IT Services & Consulting,48.6k,5.3L,2.2k,8.4k,IT Services & Consulting
7,HCLTech,3.4,IT Services & Consulting,45.8k,3.9L,232,4.5k,IT Services & Consulting
8,ICICI Bank,4.0,Banking,45.7k,1.5L,14,3k,Banking
9,Tech Mahindra,3.4,IT Services & Consulting,43.1k,2.9L,610,4.6k,IT Services & Consulting


In [21]:
final_df = pd.DataFrame()

headers = {
    "User-Agent": "Mozilla/5.0"
}

for page in range(1, 350):

    url = f"https://www.ambitionbox.com/list-of-companies?page={page}"
    response = requests.get(url, headers=headers)
    
    soup = BeautifulSoup(response.text, "html.parser")
    
    company = soup.find_all('div', class_='companyCardWrapper')

    name = []
    rating = []
    types = []
    reviews = []
    salary = []
    job = []
    interview = []

    for i in company:

        name.append(i.find('h2').get_text(strip=True))

        rating_tag = i.find('div', class_='rating_text')
        rating.append(rating_tag.get_text(strip=True) if rating_tag else None)

        type_tag = i.find('span', class_='companyCardWrapper__interLinking')
        types.append(type_tag.get_text(strip=True) if type_tag else None)

        titles = i.find_all('span', class_='companyCardWrapper__ActionTitle')
        counts = i.find_all('span', class_='companyCardWrapper__ActionCount')

        r = s = j = iv = None

        for t, c in zip(titles, counts):
            key = t.get_text(strip=True)
            value = c.get_text(strip=True)

            if 'Reviews' in key:
                r = value
            elif 'Salaries' in key:
                s = value
            elif 'Jobs' in key:
                j = value
            elif 'Interviews' in key:
                iv = value

        reviews.append(r)
        salary.append(s)
        job.append(j)
        interview.append(iv)

    d = {
        'name': name,
        'rating': rating,
        'types': types,
        'reviews': reviews,
        'salary': salary,
        'jobs': job,
        'interviews': interview
    }

    df = pd.DataFrame(d)
    final_df = pd.concat([final_df, df], ignore_index=True)


In [22]:
def convert(t):
    if isinstance(t, str) and '|' in t:
        return t.split('|')[1].strip()
    return None

def convertTypes(a):
    if isinstance(a, str) and '|' in a:
        return a.split('|')[0].strip()
    return None

final_df['Locations'] = final_df['types'].apply(convert)
final_df['types'] = final_df['types'].apply(convertTypes)

In [23]:
final_df.head()

,name,rating,types,reviews,salary,jobs,interviews,Locations
0,TCS,3.3,IT Services & Consulting,1.1L,10.1L,4.4k,11.8k,Bengaluru +441 other locations
1,Accenture,3.7,IT Services & Consulting,73.1k,6.9L,36k,9.3k,Bengaluru +259 other locations
2,Wipro,3.6,IT Services & Consulting,64.8k,4.9L,371,6.8k,Hyderabad +373 other locations
3,Cognizant,3.7,IT Services & Consulting,61.2k,6.1L,469,6.4k,Hyderabad +231 other locations
4,Capgemini,3.6,IT Services & Consulting,53k,4.9L,2.1k,5.5k,Bengaluru +180 other locations


In [24]:
final_df.to_csv('Companies.csv')